# QBronze 1.5 — Algoritmo de búsqueda de Grover

Este cuaderno desarrolla los ejercicios principales sobre el algoritmo de búsqueda de Grover.

El problema abstracto es:

$$
\text{Dado un espacio de }N\text{ elementos, encontrar un elemento marcado.}
$$

Si

$$
N=2^n,
$$

se necesitan $n$ qubits para representar los elementos. Grover ofrece una mejora cuadrática: el número de consultas al oráculo escala como

$$
O(\sqrt N),
$$

no como $O(N)$. No es una mejora exponencial.



In [ ]:
import numpy as np
from math import sqrt, pi, cos, sin, log2, floor
from collections import Counter

np.set_printoptions(precision=6, suppress=True)

ket0 = np.array([1.0, 0.0])
ket1 = np.array([0.0, 1.0])

I = np.eye(2)
X = np.array([[0.0, 1.0], [1.0, 0.0]])
Z = np.array([[1.0, 0.0], [0.0, -1.0]])
H = (1 / sqrt(2)) * np.array([[1.0, 1.0], [1.0, -1.0]])

def tensor(*vectors_or_matrices):
    """Producto tensorial en el orden escrito: A ⊗ B ⊗ C."""
    result = np.array([1.0])
    for item in vectors_or_matrices:
        result = np.kron(result, item)
    return result

def norm2(state):
    """Norma al cuadrado: suma |amplitud|^2."""
    return float(np.vdot(state, state).real)

def probs(state):
    """Probabilidades de medición en la base computacional."""
    return np.abs(state) ** 2

def basis_state(bitstring):
    """Vector base |bitstring⟩ en orden lógico de izquierda a derecha."""
    n = len(bitstring)
    index = int(bitstring, 2)
    v = np.zeros(2**n)
    v[index] = 1.0
    return v

def bitstrings(n):
    return [format(i, f"0{n}b") for i in range(2**n)]

def show_state(state, n=None, tol=1e-10):
    """Representación textual compacta de un vector de estado."""
    if n is None:
        n = int(round(log2(len(state))))
    terms = []
    for amp, bits in zip(state, bitstrings(n)):
        if abs(amp) > tol:
            terms.append(f"{amp:+.6g}|{bits}⟩")
    return " ".join(terms) if terms else "0"

def sample_counts(state, shots=1024, seed=7, qiskit_order=False):
    """Muestreo de una distribución de medición. Si qiskit_order=True, invierte las etiquetas."""
    rng = np.random.default_rng(seed)
    p = probs(state)
    n = int(round(log2(len(state))))
    outcomes = rng.choice(2**n, p=p/p.sum(), size=shots)
    labels = []
    for i in outcomes:
        bits = format(i, f"0{n}b")
        labels.append(bits[::-1] if qiskit_order else bits)
    return dict(sorted(Counter(labels).items()))

def is_valid_quantum_state(v, tol=1e-9):
    return abs(norm2(np.asarray(v, dtype=float)) - 1.0) < tol

def uniform_state(N):
    return np.ones(N) / sqrt(N)

def phase_oracle(state, marked_indices):
    """Invierte el signo de los elementos marcados."""
    out = state.copy()
    for idx in marked_indices:
        out[idx] *= -1
    return out

def diffusion(state):
    """Inversión sobre el promedio: a_i -> 2*promedio - a_i."""
    mean = np.mean(state)
    return 2*mean - state

def grover_iteration(state, marked_indices):
    return diffusion(phase_oracle(state, marked_indices))

def label_to_index(label):
    return int(label, 2)

def index_to_label(index, n):
    return format(index, f"0{n}b")

## 1. Qué sí afirma Grover

Afirmaciones correctas:

- Grover da una mejora cuadrática para búsqueda no estructurada:

$$
O(N)\longrightarrow O(\sqrt N).
$$

Afirmaciones incorrectas:

- No da una mejora exponencial.
- No mejora indefinidamente al aumentar el número de iteraciones. Después del punto óptimo, la probabilidad del elemento marcado puede volver a disminuir.
- No se necesita conocer internamente qué elemento está marcado; se usa un oráculo que implementa la marca.

La idea central es alternar dos operaciones:

1. Oráculo: cambia el signo de los elementos marcados.
2. Difusión: inversión de amplitudes alrededor del promedio.

In [ ]:
N = 32
print("Búsqueda clásica no estructurada: O(N) =", N)
print("Escala de Grover: O(sqrt(N)) ≈", sqrt(N))
print("Grover es cuadrático, no exponencial.")

## 2. Phase kickback: marcar sin revelar el elemento marcado

El oráculo booleano se puede escribir como

$$
B_f|x\rangle|b\rangle=|x\rangle|b\oplus f(x)\rangle.
$$

Si el segundo registro se prepara como

$$
|-\rangle=\frac{|0\rangle-|1\rangle}{\sqrt{2}},
$$

entonces ocurre:

$$
B_f|x\rangle|-\rangle=(-1)^{f(x)}|x\rangle|-\rangle.
$$

Así, si $f(x)=1$, el estado $|x\rangle$ recibe un signo negativo. Si $f(x)=0$, queda igual. Esto permite marcar el elemento sin medirlo ni conocerlo explícitamente dentro del algoritmo.

In [ ]:
# Ejemplo con N=4 y elemento marcado |01>.
n = 2
N = 2**n
state = uniform_state(N)
marked = [label_to_index("01")]
state_after_oracle = phase_oracle(state, marked)

print("Estado inicial uniforme:")
for bits, amp in zip(bitstrings(n), state):
    print(bits, amp)

print("\nDespués del oráculo que marca |01>:")
for bits, amp in zip(bitstrings(n), state_after_oracle):
    print(bits, amp)

## 3. Número de qubits para representar 32 elementos

Si el espacio de búsqueda tiene $32$ elementos, buscamos $n$ tal que

$$
2^n=32.
$$

Como

$$
32=2^5,
$$

se necesitan 5 qubits para representar los elementos. No se cuentan aquí qubits auxiliares.

In [ ]:
N = 32
n = int(log2(N))
print("N =", N)
print("n = log2(N) =", n)
print("Verificación: 2^n =", 2**n)

## 4. Crear superposición uniforme de 32 elementos

Para $n=5$ qubits, el estado inicial es

$$
|00000\rangle.
$$

Aplicar $H$ a cada qubit produce

$$
H^{\otimes 5}|00000\rangle
=
\frac{1}{\sqrt{32}}\sum_{x=0}^{31}|x\rangle.
$$

En código de circuito, conceptualmente:

```python
for i in range(5):
    mycircuit.h(qreg[i])
```

Por tanto, en el fragmento

```python
for i in range(a):
    mycircuit.b(qreg[i])
```

se usa

$$
a=5,
\qquad
b=h.
$$

In [ ]:
N = 32
state = uniform_state(N)
print("amplitud esperada de cada elemento:", 1/sqrt(N))
print("norma^2:", norm2(state))
print("primeros 8 elementos:")
for i in range(8):
    print(index_to_label(i, 5), state[i])

## 5. Preparar el ancilla en $|-\rangle$

Grover suele usar un qubit auxiliar en el estado

$$
|-\rangle=\frac{|0\rangle-|1\rangle}{\sqrt{2}}.
$$

Desde $|0\rangle$, se prepara así:

$$
|0\rangle \xrightarrow{X} |1\rangle
\quad\text{y}\quad
|1\rangle \xrightarrow{H} \frac{|0\rangle-|1\rangle}{\sqrt{2}}.
$$

Por tanto, se aplica primero $X$ y luego $H$.

In [ ]:
ancilla = ket0.copy()
ancilla = X @ ancilla
ancilla = H @ ancilla
print("ancilla:", show_state(ancilla, n=1))
print("vector:", ancilla)

## 6. Identificar el elemento marcado por el signo negativo

Para 4 elementos, el estado uniforme es

$$
\frac{|00\rangle+|01\rangle+|10\rangle+|11\rangle}{2}.
$$

Después de la fase de consulta, si el estado es

$$
\frac{|00\rangle-|01\rangle+|10\rangle+|11\rangle}{2},
$$

el único término con signo negativo es $|01\rangle$. Por tanto, el elemento marcado es

$$
|01\rangle.
$$

In [ ]:
state = np.array([1/2, -1/2, 1/2, 1/2])
labels = bitstrings(2)
for bits, amp in zip(labels, state):
    print(bits, amp)
marked = [bits for bits, amp in zip(labels, state) if amp < 0]
print("elemento marcado por signo negativo:", marked)

## 7. Inversión sobre el promedio

La fase de difusión de Grover se interpreta como inversión sobre el promedio.

Si las amplitudes son

$$
a_0,a_1,\ldots,a_{N-1},
$$

y el promedio es

$$
\bar a=\frac1N\sum_i a_i,
$$

entonces cada amplitud se transforma como

$$
a_i' = 2\bar a - a_i.
$$

Geométricamente, esto es una reflexión respecto al estado de superposición uniforme.

In [ ]:
amplitudes = np.array([0.5, -0.5, 0.5, 0.5])
mean = np.mean(amplitudes)
reflected = diffusion(amplitudes)
print("amplitudes:", amplitudes)
print("promedio:", mean)
print("después de inversión sobre el promedio:", reflected)

## 8. Una iteración completa para 4 elementos

Tomemos $N=4$ y elemento marcado $|01\rangle$.

Estado inicial:

$$
|s\rangle=\frac{|00\rangle+|01\rangle+|10\rangle+|11\rangle}{2}.
$$

Después del oráculo:

$$
\frac{|00\rangle-|01\rangle+|10\rangle+|11\rangle}{2}.
$$

El promedio de las amplitudes es

$$
\bar a=\frac{1/2-1/2+1/2+1/2}{4}=\frac14.
$$

Aplicamos

$$
a_i'=2\bar a-a_i.
$$

Para el marcado:

$$
a'_{01}=2\cdot\frac14-\left(-\frac12\right)=1.
$$

Para un no marcado:

$$
a'=2\cdot\frac14-\frac12=0.
$$

Resultado:

$$
|01\rangle.
$$

Con 4 elementos y 1 marcado, una iteración basta para concentrar toda la probabilidad en el elemento correcto.

In [ ]:
n = 2
N = 4
state = uniform_state(N)
marked = [label_to_index("01")]
print("inicial:", show_state(state, n=n))

state = phase_oracle(state, marked)
print("después del oráculo:", show_state(state, n=n))

state = diffusion(state)
print("después de difusión:", show_state(state, n=n))
print("probabilidades:")
for bits, prob in zip(bitstrings(n), probs(state)):
    print(bits, prob)

## 9. Más iteraciones no siempre significan mejor resultado

Grover es una rotación en un subespacio bidimensional: el eje de elementos marcados y el eje de elementos no marcados. Si se itera demasiado, el estado puede pasar de largo el punto óptimo.

Para un marcado entre $N$ elementos, una aproximación común para el número de iteraciones es

$$
k\approx \left\lfloor \frac{\pi}{4}\sqrt N\right\rfloor.
$$

Para $N=32$:

$$
\frac{\pi}{4}\sqrt{32}\approx 4.44.
$$

Se esperan alrededor de 4 iteraciones, dependiendo de la convención de redondeo y del número de soluciones marcadas.

In [ ]:
N = 32
estimate = (pi/4) * sqrt(N)
print("estimación continua:", estimate)
print("floor:", floor(estimate))
print("round:", round(estimate))

# Simulación para mostrar que la probabilidad sube y luego baja.
n = 5
marked = [17]
state = uniform_state(N)
for k in range(10):
    p_marked = probs(state)[marked[0]]
    print(f"iteración {k}: P(marcado) = {p_marked:.6f}")
    state = grover_iteration(state, marked)

## 10. Simular Grover para 32 elementos

Elegimos arbitrariamente el elemento marcado 17. En binario de 5 bits:

$$
17=10001_2.
$$

La simulación muestra cómo la probabilidad del elemento marcado aumenta durante las primeras iteraciones.

In [ ]:
N = 32
n = 5
marked_index = 17
marked_label = index_to_label(marked_index, n)
state = uniform_state(N)

print("elemento marcado:", marked_index, "=", marked_label)
for k in range(6):
    probs_now = probs(state)
    most_likely = int(np.argmax(probs_now))
    print(f"k={k}: P(marcado)={probs_now[marked_index]:.6f}, más probable={index_to_label(most_likely,n)}")
    state = grover_iteration(state, [marked_index])

## 11. Oráculo como caja negra

El algoritmo no necesita conocer internamente cómo se implementa el oráculo. Sólo requiere que el oráculo produzca el efecto correcto:

$$
|x\rangle\mapsto (-1)^{f(x)}|x\rangle.
$$

Si $f(x)=1$, el signo cambia. Si $f(x)=0$, el signo no cambia.

Este es el mecanismo que permite que el elemento marcado sea distinguido por interferencia de amplitudes, no por inspección clásica directa.

In [ ]:
def black_box_f(x):
    # La implementación interna podría estar oculta.
    # Aquí la escribimos sólo para simular el comportamiento.
    return 1 if x == 17 else 0

def black_box_phase_oracle(state):
    out = state.copy()
    for x in range(len(out)):
        if black_box_f(x) == 1:
            out[x] *= -1
    return out

state = uniform_state(32)
after = black_box_phase_oracle(state)
negative_indices = [i for i, amp in enumerate(after) if amp < 0]
print("índices con signo negativo:", negative_indices)
print("etiqueta binaria:", [index_to_label(i, 5) for i in negative_indices])

## 12. Resumen operativo del módulo

Reglas clave:

$$
N=2^n \quad\Longrightarrow\quad n=\log_2(N)\text{ qubits}.
$$

$$
H^{\otimes n}|0\cdots0\rangle=\frac1{\sqrt N}\sum_{x=0}^{N-1}|x\rangle.
$$

$$
B_f|x\rangle|-\rangle=(-1)^{f(x)}|x\rangle|-\rangle.
$$

$$
\text{difusión: } a_i' = 2\bar a-a_i.
$$

Grover alterna oráculo y difusión para amplificar la probabilidad del elemento marcado.